<a href="https://colab.research.google.com/github/ttobias28/PhishRadar-Project/blob/distilBert/PhishRadar_DistilBERT_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhishRadar — Approach 2: Fine-tuning DistilBERT for phishing detection

**Data: the team's cross-source splits** (`phishradar_splits.zip`). The model trains on **Enron, Ling and SpamAssassin** only. It is then tested on:

| Test set | What it tells you |
|---|---|
| `test_insource` | Same sources as training. The "normal" score. |
| `test_ceas` | A source it never saw (spam + legit). Does it generalize? |
| `test_nazario_paired` | Real phishing (Nazario) mixed 50/50 with unseen legit emails (CEAS). The closest thing to a real phishing test. |
| `test_nazario` | Real phishing only. Shows the detection rate (recall). |
| `test_nigerian` | Advance-fee scams only. Shows the detection rate (recall). |

**Before you start (one time):** unzip `phishradar_splits.zip` and upload all the `.csv` files into **My Drive → PhishRadar → data** in Google Drive. If old CSVs from the first dataset are in that folder, delete them first so they don't get mixed up.

**Every time (Google Colab):**
1. Open this notebook in Colab (colab.research.google.com → **File → Upload notebook**, or open it from your Drive).
2. **Runtime → Change runtime type → T4 GPU**.
3. Run the cells top to bottom with **Shift+Enter**.

**Running in plain Jupyter instead?** Put the CSVs in a folder called `data` next to this notebook. The notebook detects that it isn't in Colab and skips the Google Drive steps. Training without a GPU is very slow, though, so do the actual training in Colab.

**First run:** `SAMPLE_SIZE` trains on 5,000 emails so the run finishes in about 15 minutes. Once it works, set it to `None` to train on all ~30,000 training emails.

## 1. Install libraries
- `transformers`: Hugging Face's library for BERT and similar models
- `datasets`: efficient data handling for training
- `accelerate`: what the training loop uses under the hood

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn seaborn

## 2. Imports and GPU check

In [ ]:
import os, re, sys, glob, zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix, classification_report)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! In Colab: Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 3. Settings
These are the knobs you'll tweak. Changing them and comparing results is good material for your report.
- **MODEL_NAME**: which pre-trained model to start from. Try `"bert-base-uncased"` later to compare with full BERT.
- **MAX_LENGTH**: how many tokens (word pieces) of each email the model reads. BERT's hard limit is 512.
- **EPOCHS**: how many full passes over the training data. Fine-tuning usually needs only 2–4.
- **LEARNING_RATE**: how big each update step is. 2e-5 is the standard for BERT fine-tuning.
- **SEED**: fixes the randomness so results are reproducible.
- **SAMPLE_SIZE**: how many training emails to use. Small = fast test run; `None` = everything.
- **NORMALIZE_SPACING**: fixes a formatting quirk explained in step 5. Try a run with `False` to see how much it matters.

In [ ]:
MODEL_NAME        = "distilbert-base-uncased"
MAX_LENGTH        = 256
EPOCHS            = 2
LEARNING_RATE     = 2e-5
BATCH_SIZE        = 16
SEED              = 42
SAMPLE_SIZE       = 5000    # set to None for the full training set
NORMALIZE_SPACING = True

DATA_DIR = "/content/drive/MyDrive/PhishRadar/data" if IN_COLAB else "data"
OUT_DIR  = "/content/drive/MyDrive/PhishRadar" if IN_COLAB else "."
TEST_SETS = ["test_insource", "test_ceas", "test_nazario_paired", "test_nazario", "test_nigerian"]

## 4. Load the data
Each file has three columns: `text` (subject + body), `label` (1 = phishing/spam, 0 = legit) and `source` (which original collection it came from).

In Colab, this cell connects your Google Drive (click through the permission pop-up). If it can't find the files there, it shows an upload button. Pick `phishradar_splits.zip`.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    if not IN_COLAB:
        raise FileNotFoundError(f"Put the split CSVs in a folder called '{DATA_DIR}' next to this notebook.")
    print(f"Couldn't find the CSVs in {DATA_DIR}. Upload phishradar_splits.zip instead:")
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(n for n in uploaded if n.endswith(".zip"))
    zipfile.ZipFile(zip_name).extractall("/content/data")
    DATA_DIR = os.path.dirname(glob.glob("/content/data/**/train.csv", recursive=True)[0])

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
tests    = {name: pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv")) for name in TEST_SETS}

print(f"{'train':>20}: {len(train_df):>6,} emails, {train_df.label.mean():.0%} phishing")
print(f"{'val':>20}: {len(val_df):>6,} emails, {val_df.label.mean():.0%} phishing")
for name, d in tests.items():
    print(f"{name:>20}: {len(d):>6,} emails, {d.label.mean():.0%} phishing")

## 5. Fix a formatting quirk (spaced punctuation)
Enron and Ling, which make up most of the training data, were stored with spaces around punctuation: `doesn ' t , and 6 / 10 / 00`. SpamAssassin and every *test* source use normal punctuation, and SpamAssassin is mostly legit. Without a fix, BERT could learn "normal punctuation means legit," and that shortcut would hurt it on every test set.

`normalize_spacing` removes those extra spaces. It's applied the same way to every split, so it's fair. Run it once with `NORMALIZE_SPACING = False` and compare. The difference is a nice finding for your report.

In [ ]:
def normalize_spacing(t):
    t = re.sub(r"(\w) ' (s|t|re|ve|ll|d|m)\b", r"\1'\2", t)   # doesn ' t -> doesn't
    t = re.sub(r"\s+([,.;:!?%)\]}])", r"\1", t)                # word , -> word,
    t = re.sub(r"([(\[{$])\s+", r"\1", t)                      # ( word -> (word
    t = re.sub(r"(\d) ([.,:/]) (\d)", r"\1\2\3", t)             # 6 / 10 -> 6/10
    t = re.sub(r"(\d)([.,:/]) (\d)", r"\1\2\3", t)
    return t

example = train_df[train_df.source == "Enron"].text.iloc[0][:200]
print("before:", example)
print("after: ", normalize_spacing(example))

if NORMALIZE_SPACING:
    for d in [train_df, val_df, *tests.values()]:
        d["text"] = d["text"].map(normalize_spacing)

## 6. Look at the data
Always look before you train.

In [ ]:
pd.crosstab(train_df["source"], train_df["label"].map({0: "legit", 1: "phishing/spam"})).plot.barh(stacked=True)
plt.title("Training emails by source"); plt.xlabel("emails"); plt.show()

print("Example phishing/spam email:\n", train_df[train_df.label == 1].text.iloc[0][:400])
print("\nExample legit email:\n", train_df[train_df.label == 0].text.iloc[0][:400])

## 7. Optional: shrink for a quick test run
`SAMPLE_SIZE` takes a random subset of train and validation. **The test sets are never shrunk**, so your final scores are always on the same emails your teammates use.

In [ ]:
if SAMPLE_SIZE:
    train_df = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), random_state=SEED)
    val_df   = val_df.sample(n=min(SAMPLE_SIZE // 5, len(val_df)), random_state=SEED)
print(f"Training on {len(train_df):,} emails, validating on {len(val_df):,}")

## 8. Peek at tokenization
BERT doesn't read words. It reads **tokens**, which are word pieces mapped to ID numbers. Rare words get split up: "unverified" might become `un`, `##ver`, `##ified`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "URGENT: Your account has been suspended. Verify your password immediately at http://secure-login.xyz"
print(tokenizer.tokenize(example))
print(tokenizer(example)["input_ids"])

## 9. Tokenize every email
`truncation=True` cuts emails off at `MAX_LENGTH` tokens. Padding (filling short emails with blanks so a batch lines up) happens later, batch by batch.

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

def to_dataset(frame):
    ds = Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    return ds.map(tokenize, batched=True, remove_columns=["text"])

train_ds, val_ds = to_dataset(train_df), to_dataset(val_df)
test_ds = {name: to_dataset(d) for name, d in tests.items()}
train_ds

## 10. Load the model
`AutoModelForSequenceClassification` = pre-trained DistilBERT **plus a fresh classification head** with 2 outputs.

You'll see a warning that some weights are "newly initialized." **That's expected.** It's the new head, which gets learned during training.

In [ ]:
id2label = {0: "legit", 1: "phishing"}
label2id = {"legit": 0, "phishing": 1}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)

## 11. Metrics
- **Precision**: of the emails flagged as phishing, how many really were.
- **Recall**: of the real phishing emails, how many we caught.
- **F1**: a balance of precision and recall.
- **False-positive rate**: how often legit emails get wrongly flagged.

In [ ]:
def score(labels, preds):
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1,
            "false_positive_rate": fp / (fp + tn) if (fp + tn) else float("nan")}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return score(labels, np.argmax(logits, axis=-1))

## 12. Train!
The `Trainer` runs the training loop: feed batches in, measure the error (loss), nudge the weights, repeat. Watch the **training loss go down**; that means it's learning. After each epoch it scores the validation set and keeps the checkpoint with the best F1.

In [ ]:
args = TrainingArguments(
    output_dir="phishradar-distilbert",
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),   # mixed precision = faster on GPU
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 13. Score every test set
These are the numbers for your report. `test_ceas` has ~39,000 emails, so this takes a few minutes.

For `test_nazario` and `test_nigerian` (phishing only), **recall is the detection rate**. Precision is trivially 1.0 and the false-positive rate doesn't exist there, so use `test_nazario_paired` for those.

In [ ]:
rows, all_preds = [], []
for name, d in tests.items():
    out = trainer.predict(test_ds[name])
    preds = np.argmax(out.predictions, axis=-1)
    probs = torch.softmax(torch.tensor(out.predictions).float(), dim=-1)[:, 1].numpy()
    m = score(d["label"].values, preds)
    if d["label"].nunique() == 1:                      # phishing-only set
        m.update(precision=float("nan"), f1=float("nan"))
    rows.append({"test_set": name, "emails": len(d), **m})
    all_preds.append(d.assign(test_set=name, row=np.arange(len(d)), pred=preds, phish_prob=probs))

results_table = pd.DataFrame(rows).set_index("test_set").round(4)
predictions = pd.concat(all_preds, ignore_index=True)
results_table

## 14. Confusion matrices
Rows are the true answer, columns are the model's guess. The off-diagonal boxes are the mistakes.

In [ ]:
mixed = [n for n in TEST_SETS if tests[n]["label"].nunique() == 2]
fig, axes = plt.subplots(1, len(mixed), figsize=(5 * len(mixed), 4))
for ax, name in zip(np.atleast_1d(axes), mixed):
    p = predictions[predictions.test_set == name]
    sns.heatmap(confusion_matrix(p.label, p.pred, labels=[0, 1]), annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["legit", "phishing"], yticklabels=["legit", "phishing"], cbar=False)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

p = predictions[predictions.test_set == "test_insource"]
print("In-source test, by source:")
print(p.assign(correct=p.pred == p.label).groupby("source").correct.agg(emails="size", accuracy="mean").round(4))

## 15. Error analysis
Your proposal promises to look at *which* phishing patterns each method misses.
- **False negatives**: phishing emails the model let through (the dangerous kind).
- **False positives**: legit emails it wrongly flagged.

**Files for the team** (small enough for GitHub):
- `bert_predictions.csv`: one row per test email with `test_set`, `row` (the email's position in that test file, starting at 0), `label`, `pred` and `phish_prob`. No email text, so it stays small. Teammates match emails by `test_set` + `row`.
- `bert_results.csv`: the scores table from step 13.
- `bert_errors.csv`: only the mistakes, *with* email text, for reading.

In [ ]:
predictions[["test_set", "row", "label", "source", "pred", "phish_prob"]].to_csv("bert_predictions.csv", index=False)
results_table.to_csv("bert_results.csv")

errors = predictions[predictions.pred != predictions.label].copy()
errors["error_type"] = np.where(errors.label == 1, "false_negative", "false_positive")
errors[["test_set", "row", "error_type", "label", "pred", "phish_prob", "source", "text"]].to_csv("bert_errors.csv", index=False)
print(pd.crosstab(errors.test_set, errors.error_type))

pd.set_option("display.max_colwidth", 300)
# Real phishing the model missed, most confident mistakes first:
errors[errors.test_set == "test_nazario"].sort_values("phish_prob")[["phish_prob", "text"]].head(10)

## 16. Try it on your own email
Paste in any text. This is also how you'll test the AI-generated adversarial emails from your stretch goal.

In [ ]:
def predict(text):
    if NORMALIZE_SPACING:
        text = normalize_spacing(text)
    model.eval()
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    return {"legit": round(probs[0].item(), 4), "phishing": round(probs[1].item(), 4)}

print(predict("Hi team, attached are the slides for Thursday's meeting. Let me know if you have questions."))
print(predict("Your PayPal account is limited. Click here within 24 hours to restore access or it will be closed."))

## 17. Save your trained model and results
Colab erases everything when the session ends. In Colab this saves to **My Drive → PhishRadar**; in plain Jupyter it saves next to the notebook.

In [ ]:
import shutil
save_dir = os.path.join(OUT_DIR, "distilbert-model")
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
if os.path.abspath(OUT_DIR) != os.path.abspath("."):
    for f in ["bert_predictions.csv", "bert_errors.csv", "bert_results.csv"]:
        shutil.copy(f, OUT_DIR)
print("Saved model to", save_dir, "and result files to", OUT_DIR)

# Later, reload it with:
# model = AutoModelForSequenceClassification.from_pretrained(save_dir)
# tokenizer = AutoTokenizer.from_pretrained(save_dir)

## Next experiments (good report material)
1. Set `SAMPLE_SIZE = None` and train on the full training set. How much do the cross-source scores improve?
2. Set `NORMALIZE_SPACING = False`. Does the model do worse on CEAS and Nazario? That would mean it had learned a formatting shortcut.
3. Change `MAX_LENGTH` to 128 or 512. Does seeing more of the email help?
4. Swap in `"bert-base-uncased"`. Is full BERT worth the extra training time?
5. Compare the in-source score to the cross-source scores. A big drop means the model learned the training collections, not phishing in general.